# Reverse RAG: checking a medical device QMS against the EU MDR

**Declan Lenahan, DL Consultancy** · [Code on GitHub](https://github.com/Declan444/qms-gap-rag)

Every audit in medical devices comes down to one question: *does our quality system actually meet the regulation?* Answering it means reading every procedure against every requirement, by hand.

This notebook tests whether a **small, free AI model** can do a first pass. It uses retrieval-augmented generation (RAG), run in reverse:

- In normal RAG, a person asks a question and the system finds the matching documents.
- Here, **each regulatory requirement is the question**. The system finds the company procedures closest in meaning, and a local model judges: **COVERED, WEAK or MISSING**, citing the procedure section it relied on.

**The test.** Slieve Bloom Medical Ltd is a fictional company with eight procedures. I planted fifteen problems in them: missing procedures, wrong reporting deadlines, outdated standards, and two procedures that contradict each other. Because I know where every gap is, the tool can be scored.

**The rule.** Everything the tool produces is a draft for a qualified person to review. It flags; people decide.

---
**To run it:** Runtime › Change runtime type › **T4 GPU**, then run each cell in order. About 10 minutes in total, most of it downloading the model.

## 1. Get the project
Downloads the code, the eight procedures, the checklist and the answer key from GitHub.

In [ ]:
import os
!rm -rf /content/qms-gap-rag
!git clone -q https://github.com/Declan444/qms-gap-rag.git /content/qms-gap-rag
os.chdir('/content/qms-gap-rag')
print(sorted(os.listdir('data/procedures')))

## 2. Install Ollama and start it

[Ollama](https://ollama.com) runs open AI models on your own machine. Here it runs on Colab's GPU; on a company PC the same code runs with nothing leaving the building.

In [ ]:
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time
server = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
!ollama --version

## 3. Download the model

**Qwen2.5 7B Instruct**, an open model from Alibaba, about 5 GB. Small by today's standards, and free to run.

In [ ]:
!ollama pull qwen2.5:7b-instruct

## 4. Install the Python libraries

In [ ]:
!pip -q install -r requirements.txt

## 5. Settings

Each setting is one thing you can change and rescore. The lecture that inspired this showed a single setting fixing one answer and breaking another, so change one at a time.

- `TOP_K`: how many procedure extracts the model reads per requirement
- `MAX_CHUNK_WORDS`: how finely the procedures are cut up (rerun step 6 if you change it)
- `MIN_SIMILARITY`: below this, nothing counts as relevant, and the requirement is marked MISSING without asking the model

In [ ]:
import re
settings = {
    'LLM_MODEL': '"qwen2.5:7b-instruct"',
    'TOP_K': '4',
    'MAX_CHUNK_WORDS': '120',
    'MIN_SIMILARITY': '0.30',
}
text = open('config.py').read()
for k, v in settings.items():
    text = re.sub(rf'^{k} = .*?(\s+#.*)?$', lambda m: f'{k} = {v}' + (m.group(1) or ''), text, flags=re.M)
open('config.py', 'w').write(text)
print('\n'.join(l for l in text.splitlines() if l.split(' =')[0] in settings))

## 6. Build the index (RAG step 1)

The procedures are cut into chunks at their own headings, and each chunk becomes 384 numbers that stand for its meaning (an *embedding*). Chunks close in meaning end up close in numbers.

In [ ]:
!python build_index.py

## 7. Run the gap check (RAG steps 2 and 3, in reverse)

For each of the 13 requirements in MDR Article 10(9): find the closest procedure chunks, then ask the model for a verdict with citations.

In [ ]:
!python gap_report.py

## 8. Score it against the answer key

The answer key lists the fifteen planted problems. A verdict is marked right if it matches what a careful auditor would say.

In [ ]:
!python score.py

## 9. Read the report

This is what a quality manager would receive: a verdict, a reason, a suggested fix, and the procedure sections to open and check.

In [ ]:
from IPython.display import Markdown, display
latest = sorted(f for f in os.listdir('output') if f.startswith('gap_report_') and f.endswith('.md'))[-1]
display(Markdown(open(f'output/{latest}').read()))

## What to look for

- **Missing procedures** (UDI, clinical evaluation): did the similarity threshold catch them, or did the model accept loosely related text as coverage?
- **Wrong rules** (serious incidents "within 30 days"; a Class I report used for a Class IIa device): the model only sees the company's own procedures, not the regulation, so it may not know these are wrong. That is the case for stage 3: adding the regulation itself as a second source.
- **Citations**: open each one. A right verdict for the wrong reason is only half right.

Expect it to get some wrong. That is the point: a tool you can test is a tool you can trust, or know not to.

## 10. Download the results
Colab deletes everything when the session ends. Keep each run's report and score.

In [ ]:
from google.colab import files
!zip -qr results.zip output -x 'output/index/*'
files.download('results.zip')